In [1]:
# تابع لود دیتاست
import zipfile
import pandas as pd

def load_sentiment_zip(zip_path):
    texts = []
    labels = []

    with zipfile.ZipFile(zip_path, "r") as z:
        for name in z.namelist():
            if not name.endswith(".txt"):
                continue

            if "/pos/" in name:
                label = 1
            elif "/neg/" in name:
                label = 0
            else:
                continue

            txt = z.read(name).decode("utf-8", errors="replace").strip()
            texts.append(txt)
            labels.append(label)

    return pd.DataFrame({"text": texts, "label": labels})


In [2]:
# لود دیتاست و چاپ خروجی
zip_path = "/content/Sentiment Labeled Dataset.zip"
df = load_sentiment_zip(zip_path)

print("len(df):", len(df))
print(df["label"].value_counts())
df.head()


len(df): 25000
label
0    12500
1    12500
Name: count, dtype: int64


,text,label
0,This film lacked something I couldn't put my f...,0
1,This film is mediocre at best. Angie Harmon is...,0
2,The characters are unlikeable and the script i...,0
3,I had a lot of expectations from this movie an...,0
4,I had never thought the standard of Yashraj fi...,0


In [3]:
# تقسیم داده، TF-IDF و آموزش به مدل
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# جدا کردن ویژگی‌ها و برچسب‌ها
X = df["text"]
y = df["label"]

# تقسیم به train و test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ساخت TF-IDF
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ساخت و آموزش مدل
model = LogisticRegression(max_iter=300)
model.fit(X_train_tfidf, y_train)

# ارزیابی مدل
y_pred = model.predict(X_test_tfidf)

print("Accuracy on test data:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy on test data: 0.8852

Classification Report:
              precision    recall  f1-score   support

    negative       0.89      0.87      0.88      2500
    positive       0.88      0.90      0.89      2500

    accuracy                           0.89      5000
   macro avg       0.89      0.89      0.89      5000
weighted avg       0.89      0.89      0.89      5000


Confusion Matrix:
[[2187  313]
 [ 261 2239]]


In [4]:
# تابع با پیام احساسات مثبت و منفی
def predict_and_print_sentiment(text: str):
    """
    یک جمله می‌گیرد، آن را تبدیل به TF-IDF می‌کند
    و بر اساس خروجی مدل، پیام احساسی مناسب را چاپ می‌کند.
    """
    X_vec = vectorizer.transform([text])
    pred = model.predict(X_vec)[0]   # 0 یا 1

    if pred == 1:
        print("This sentence has a positive feeling 😄")
    else:
        print("This sentence has a negative feeling 😞")


In [5]:
# ورودی از کاربر بگیر تا زمانی که خروج را بزند
print("Interactive sentiment checker is ready.")
print("Type a sentence in English to analyze its feeling.")
print("To exit, type: exit  یا  quit  یا  q  یا فقط Enter خالی.\n")

while True:
    user_input = input("Enter a sentence (or 'exit' to quit): ").strip()

    # شرط خروج
    if user_input == "" or user_input.lower() in ["exit", "quit", "q"]:
        print("Exiting sentiment checker. Goodbye!")
        break

    # پیش‌بینی و نمایش احساس جمله
    predict_and_print_sentiment(user_input)
    print("-" * 40)


Interactive sentiment checker is ready.
Type a sentence in English to analyze its feeling.
To exit, type: exit  یا  quit  یا  q  یا فقط Enter خالی.

Enter a sentence (or 'exit' to quit): I love this movie
This sentence has a positive feeling 😄
----------------------------------------
Enter a sentence (or 'exit' to quit): This movie was great
This sentence has a positive feeling 😄
----------------------------------------
Enter a sentence (or 'exit' to quit): This movie was boring
This sentence has a negative feeling 😞
----------------------------------------
Enter a sentence (or 'exit' to quit): I don't like it
This sentence has a negative feeling 😞
----------------------------------------
Enter a sentence (or 'exit' to quit): q
Exiting sentiment checker. Goodbye!
